<a href="https://www.kaggle.com/code/muzammalhussain11/yolov8n-baselines-jersey-2?scriptVersionId=371202916" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# YOLOv8n and YOLO11n baselines for jersey-number localisation

Companion to `kaggle_train.ipynb` in
[MHKhan50/jersey-number-detection](https://github.com/MHKhan50/jersey-number-detection).
It trains YOLOv8n and YOLO11n in up to seven settings and scores every run with **the same evaluator as
JerseyNet** (COCO-style AP over all boxes, first-box IoU, 95% bootstrap intervals):

| Setting | Weights | Input | Epochs | What it isolates |
|---|---|---|---|---|
| COCO-pretrained, 640 px | `yolov8n.pt` | 640 | 50 | the earlier comparison |
| COCO-pretrained, 256 px | `yolov8n.pt` | 256 | 50 | pretraining, at JerseyNet's input size |
| From scratch, 256 px | `yolov8n.yaml` | 256 | 140 | architecture only, no pretraining |
| From scratch, 384 px | `yolov8n.yaml` | 384 | 140 | matches JerseyNet's best recipe (384 px) |
| From scratch, 256 px, JerseyNet augmentation | `yolov8n.yaml` | 256 | 140 | augmentation matched to JerseyNet |
| YOLO11n, COCO-pretrained, 640 px | `yolo11n.pt` | 640 | 50 | the current nano model, pretrained |
| YOLO11n, from scratch, 384 px | `yolo11n.yaml` | 384 | 140 | the current nano model, architecture only |

Each setting runs with seeds 0, 1 and 2 (all settings for seed 0 first). `ONLY` in the
settings cell limits the run to some settings; by default it runs only the two YOLO11n
settings, as the YOLOv8n settings are already reported in the repository. Checkpoints are chosen
on validation; the test split is only used for the final scores.

**Kaggle settings:** Accelerator *GPU T4 x2*, Internet *on* (Ultralytics and the COCO weights
are downloaded), the `jersey-2` dataset attached under *Input*. Then *Save Version → Save & Run
All (Commit)*. Results go to `/kaggle/working/yolo_results`. Re-running with the previous
version's output attached as an input is not needed: unfinished runs simply start again.

In [1]:
# Settings
import os

REPO_URL = "https://github.com/MHKhan50/jersey-number-detection.git"
REPO_REF = "main"
TIME_BUDGET_HOURS = 9.5          # no new run starts after this; Kaggle stops at 12 h
SEEDS = [0, 1, 2]
ONLY = ["yolo11n_coco_pretrained_640_px", "yolo11n_from_scratch_384_px"]   # [] runs every setting
OUT_DIR = "/kaggle/working/yolo_results"
INPUT_ROOTS = ["/kaggle/input"]

# Continuous-integration smoke test only: synthetic data, one tiny CPU run.
SMOKE = os.environ.get("JND_SMOKE") == "1"
if SMOKE:
    SEEDS, OUT_DIR = [0], os.environ.get("JND_OUT", "/tmp/jnd-yolo-smoke/results")
print({"REPO_REF": REPO_REF, "SEEDS": SEEDS, "TIME_BUDGET_HOURS": TIME_BUDGET_HOURS, "SMOKE": SMOKE})

{'REPO_REF': 'main', 'SEEDS': [0, 1, 2], 'TIME_BUDGET_HOURS': 9.5, 'SMOKE': False}


In [2]:
# Environment check and Ultralytics (PyTorch from the Kaggle image is kept as it is)
import subprocess
import sys

import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif not SMOKE:
    print("WARNING: no GPU. Set Accelerator to GPU T4 x2.")
# YOLO11n needs Ultralytics 8.3 or newer, so install it even if an older copy is present.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9"], check=True)
import ultralytics

print("ultralytics", ultralytics.__version__)

torch 2.11.0+cu128 | CUDA available: True
GPU: Tesla T4
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics 8.4.215


In [3]:
# Get the code
import time
from pathlib import Path

REPO_DIR = Path(os.environ.get("JND_REPO_DIR", "/tmp/jersey-number-detection"))
if not (REPO_DIR / "src" / "jersey_detect").exists():
    for attempt in range(3):
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)]).returncode == 0:
            break
        time.sleep(10)
SRC = str(REPO_DIR / "src")
sys.path.insert(0, SRC)
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

Cloning into '/tmp/jersey-number-detection'...


a95114674b681cefea988c8224e13127834fe63f Pin dependency versions



In [4]:
# Find the dataset
from jersey_detect.prepare import find_dataset

if SMOKE:
    from jersey_detect.synthetic import make_dataset
    DATA_ROOT = make_dataset("/tmp/jnd-yolo-smoke/data")
else:
    DATA_ROOT = find_dataset(INPUT_ROOTS)
if DATA_ROOT is None:
    raise RuntimeError("No dataset: attach jersey-2 under Input (folder with train/, valid/, test/)")
print("Dataset:", DATA_ROOT)

Dataset: /kaggle/input/datasets/muzammalhussain11/jersey-2/jersey-2


In [5]:
# Train and score every setting in a separate process (a crash cannot take the notebook down)
env = dict(os.environ, PYTHONPATH=SRC, PYTHONUNBUFFERED="1")
cmd = [sys.executable, "-m", "jersey_detect", "yolo", "--data", str(DATA_ROOT), "--out", OUT_DIR,
       "--hours", str(TIME_BUDGET_HOURS), "--seeds", *map(str, SEEDS)]
if ONLY and not SMOKE:
    cmd += ["--only", *ONLY]
if SMOKE:
    cmd.append("--smoke")
print(" ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
print("return code:", proc.wait())

/usr/bin/python3 -m jersey_detect yolo --data /kaggle/input/datasets/muzammalhussain11/jersey-2/jersey-2 --out /kaggle/working/yolo_results --hours 9.5 --seeds 0 1 2 --only yolo11n_coco_pretrained_640_px yolo11n_from_scratch_384_px
=== YOLO11n, COCO-pretrained, 640 px (seed 0)

Ultralytics 8.4.215 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_results/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze

In [6]:
# Results
import json

summary_path = Path(OUT_DIR) / "summary.json"
if summary_path.exists():
    print((Path(OUT_DIR) / "results.md").read_text())
    summary = json.loads(summary_path.read_text())
    if summary["errors"]:
        print("Errors:", json.dumps(summary["errors"], indent=1))
    if summary.get("skipped"):
        print("Skipped (time budget):", summary["skipped"])
else:
    print("No summary written; see the output above.")

# YOLOv8n baselines

Same evaluator as JerseyNet. Test intervals are 95% bootstrap intervals over images.

| Model | Split | AP50:95 | AP50 | AP75 | AR@100 | First-box mean IoU | First-box IoU ≥ 0.5 |
|---|---|---|---|---|---|---|---|
| YOLO11n, COCO-pretrained, 640 px (seed 0) | val | 0.435 | 0.722 | 0.472 | 0.558 | 0.574 | 69.2% |
| YOLO11n, COCO-pretrained, 640 px (seed 0) | test | 0.400 [0.348, 0.463] | 0.673 [0.606, 0.749] | 0.442 | 0.550 | 0.563 [0.497, 0.630] | 66.9% |
| YOLO11n, from scratch, 384 px (seed 0) | val | 0.400 | 0.684 | 0.434 | 0.517 | 0.554 | 67.3% |
| YOLO11n, from scratch, 384 px (seed 0) | test | 0.357 [0.309, 0.414] | 0.628 [0.554, 0.713] | 0.378 | 0.489 | 0.557 [0.494, 0.623] | 66.9% |
| YOLO11n, COCO-pretrained, 640 px (seed 1) | val | 0.428 | 0.722 | 0.458 | 0.548 | 0.568 | 68.8% |
| YOLO11n, COCO-pretrained, 640 px (seed 1) | test | 0.422 [0.371, 0.488] | 0.701 [0.630, 0.780] | 0.453 | 0.541 | 0.598 [0.530, 0.661] | 72.3% |
| YOLO11n, from scratch, 384 px (